In [1]:
pip install pandas scikit-learn nltk streamlit

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

def read_file(path):
    with open(path, 'rb') as f:          # read as binary
        return f.read().decode('utf-8', errors='ignore')

resumes = []

for file in os.listdir("resumes"):
    text = read_file(os.path.join("resumes", file))
    resumes.append(text)

print("✅ Resumes loaded:", len(resumes))


✅ Resumes loaded: 7


In [3]:
print(os.listdir("resumes"))

['Apeksha_Muluk_Resume.pdf', 'Rohan_Gaikwad.pdf', 'Roshan Ahiwale A (3).pdf', 'RutujaMunde_resume.pdf', 'Suraj_Jawale_data_analyst.pdf', 'Sushil Ambekar resume.pdf', 'VAISHNAVI DUMBRE _DATA SCIENCE.pdf']


In [4]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
job_desc = read_file("job_description.txt")

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z ]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

job_desc_clean = clean_text(job_desc)
resumes_clean = [clean_text(r) for r in resumes]


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91885\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([job_desc_clean] + resumes_clean)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(vectors[0:1], vectors[1:])[0]

In [7]:
ranked = sorted(
    enumerate(scores),
    key=lambda x: x[1],
    reverse=True
)

for idx, score in ranked:
    print(f"Resume {idx+1} → Match: {score*100:.2f}%")


Resume 5 → Match: 0.33%
Resume 6 → Match: 0.32%
Resume 3 → Match: 0.28%
Resume 7 → Match: 0.25%
Resume 2 → Match: 0.19%
Resume 4 → Match: 0.19%
Resume 1 → Match: 0.16%


In [8]:
files = os.listdir("resumes")

for idx, score in ranked:
    print(f"{files[idx]} → {score*100:.2f}% match")


Suraj_Jawale_data_analyst.pdf → 0.33% match
Sushil Ambekar resume.pdf → 0.32% match
Roshan Ahiwale A (3).pdf → 0.28% match
VAISHNAVI DUMBRE _DATA SCIENCE.pdf → 0.25% match
Rohan_Gaikwad.pdf → 0.19% match
RutujaMunde_resume.pdf → 0.19% match
Apeksha_Muluk_Resume.pdf → 0.16% match


In [9]:
skills = ["python", "machine learning", "data analysis", "nlp", "sql", "deep learning"]

def extract_skills(text):
    found = []
    for skill in skills:
        if skill in text:
            found.append(skill)
    return found

In [10]:
files = os.listdir("resumes")

print("\n--- FINAL RESULT ---")

for idx, score in ranked:
    resume_text = resumes_clean[idx]

    resume_skills = set(extract_skills(resume_text))
    jd_skills = set(extract_skills(job_desc_clean))

    missing = jd_skills - resume_skills

    print(f"\n📄 {files[idx]}")
    print(f"Match: {score*100:.2f}%")
    print("Skills Found:", resume_skills)
    print("Missing Skills:", missing)


--- FINAL RESULT ---

📄 Suraj_Jawale_data_analyst.pdf
Match: 0.33%
Skills Found: {'sql'}
Missing Skills: {'python'}

📄 Sushil Ambekar resume.pdf
Match: 0.32%
Skills Found: set()
Missing Skills: {'sql', 'python'}

📄 Roshan Ahiwale A (3).pdf
Match: 0.28%
Skills Found: {'sql'}
Missing Skills: {'python'}

📄 VAISHNAVI DUMBRE _DATA SCIENCE.pdf
Match: 0.25%
Skills Found: set()
Missing Skills: {'sql', 'python'}

📄 Rohan_Gaikwad.pdf
Match: 0.19%
Skills Found: {'sql'}
Missing Skills: {'python'}

📄 RutujaMunde_resume.pdf
Match: 0.19%
Skills Found: set()
Missing Skills: {'sql', 'python'}

📄 Apeksha_Muluk_Resume.pdf
Match: 0.16%
Skills Found: set()
Missing Skills: {'sql', 'python'}


In [11]:
print(f"Match: {score*100:.2f}%")

Match: 0.16%


In [12]:
for idx, score in ranked:
    if score > 0.2:   # threshold
        print(f"{files[idx]} → {score*100:.2f}%")

In [14]:
top_score = max(scores)

for idx, score in ranked:
    if score >= top_score * 0.8:
        status = "Shortlisted"
    else:
        status = "Rejected"
        
    print(f"{files[idx]} → {status}")

Suraj_Jawale_data_analyst.pdf → Shortlisted
Sushil Ambekar resume.pdf → Shortlisted
Roshan Ahiwale A (3).pdf → Shortlisted
VAISHNAVI DUMBRE _DATA SCIENCE.pdf → Rejected
Rohan_Gaikwad.pdf → Rejected
RutujaMunde_resume.pdf → Rejected
Apeksha_Muluk_Resume.pdf → Rejected
